In [ ]:
import pandas as pd
from quantecon.optimize.root_finding import results
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.impute import SimpleImputer
from scikeras.wrappers import KerasRegressor
import tensorflow as tf

from DataPreprocessing import smart_imputer, df_aggregating
import Models

In [ ]:
path = "/Users/zuzannasosnowska/.cache/kagglehub/datasets/krzysztofjamroz/apartment-prices-in-poland/versions/9/apartments_pl_2024_06.csv"

df = pd.read_csv(path)

df["price_per_m2"] = df["price"] / df["squareMeters"]

polish_cities_names = {'szczecin': 'Szczecin', 'gdynia': 'Gdynia', 'krakow': 'Kraków',  'poznan': 'Poznań', 'bialystok': 'Białystok',  'gdansk': 'Gdańsk', 'wroclaw': 'Wrocław', 'radom': 'Radom', 'rzeszow': 'Rzeszów', 'lodz': 'Łódź', 'katowice': 'Katowice', 'lublin': 'Lublin', 'czestochowa': 'Częstochowa', 'warszawa': 'Warszawa', 'bydgoszcz': 'Bydgoszcz'}

df['city'] = [polish_cities_names[city] for city in df['city']]

has_columns = ['hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']
for column in has_columns:
    df[column] = df[column].eq('yes').mul(1)

df = df.drop(columns=['id'], axis=1)

In [ ]:
print("Liczba duplikatów (rekordy z tych samych ogłoszeń): ", df.duplicated().sum())
df = df.drop_duplicates()
print("Liczba duplikatów po usunięciu (rekordy z tych samych ogłoszeń): ",df.duplicated().sum())

In [ ]:
print("Liczba duplikatów geograficznych (nieruchomości w tych samych lokalizacjach): ", df.duplicated(subset=['latitude', 'longitude']).sum())

In [ ]:
city_dataframes = {city: df[df["city"] == city] for city in df.city.unique()}
cities = df["city"].unique()

In [ ]:
cleaned_city_df = {}
for city, table in city_dataframes.items():
    # Preprocessing danych
    table, info = smart_imputer(table)
    table = table.drop(columns=["city"], axis=1)

    table = pd.get_dummies(
        table,
        columns=["type", "ownership", "buildingMaterial", "condition"],
        dtype=int
    )

    cleaned_city_df[city] = table

In [ ]:
for city, table in cleaned_city_df.items():
    if table.isna().sum().sum() != 0:
        print(city)

In [ ]:
print(cleaned_city_df["Katowice"].isna().any())
print(cleaned_city_df["Częstochowa"].isna().any())

In [ ]:
imputer = SimpleImputer(strategy="most_frequent")
cleaned_city_df["Katowice"]["buildYear"] = imputer.fit_transform(cleaned_city_df["Katowice"][["buildYear"]]).ravel()
cleaned_city_df["Częstochowa"]["buildYear"] = imputer.fit_transform(cleaned_city_df["Częstochowa"][["buildYear"]]).ravel()

In [ ]:
first_match_columns = ['floorCount', 'buildYear', 'centreDistance', 'poiCount',
        'schoolDistance', 'clinicDistance', 'postOfficeDistance', 'kindergartenDistance',
       'restaurantDistance', 'collegeDistance', 'pharmacyDistance', 'hasElevator',
        'hasSecurity', 'type_apartmentBuilding', 'type_blockOfFlats', 'type_tenement',
        'ownership_condominium', 'ownership_cooperative', 'buildingMaterial_brick',
       'buildingMaterial_concreteSlab', 'condition_low', 'condition_premium']

mean_match_columns = ['squareMeters', 'rooms', 'floor', 'hasParkingSpace', 'hasBalcony', 'hasStorageRoom', 'price', 'price_per_m2']

geo_city_df_agg = {}
for city, table in cleaned_city_df.items():
    geo_city_df_agg[city] = df_aggregating(
        df=table,
        grouped_by=['latitude', 'longitude'],
        aggregation={**{column: 'first' for column in first_match_columns}, **{column: 'mean' for column in mean_match_columns}}
    )

In [ ]:
def build_model(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_features,)),
        tf.keras.layers.Dense(
            30,
            activation="relu",
        ),
        tf.keras.layers.Dense(
            30,
            activation="relu",
        ),
        tf.keras.layers.Dense(
            30,
            activation="relu",
        ),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer="adam",
        loss="mse"
    )
    return model

In [30]:
models = {
    'GWR bisquare': Models.SklearnGWR(kernel='bisquare'),
    'Random Forest': RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
results = {}

for city, table in geo_city_df_agg.items():
    y1 = table['price'].values
    X = table.drop(columns=['price', 'price_per_m2', 'geometry', 'latitude', 'longitude'], axis=1)

    coords = list(zip(table.geometry.x, table.geometry.y))

    results[city] = Models.cross_validate_models(X=X[['squareMeters']], y=y1, coords=coords, models=models, n_splits=5)

In [ ]:
cv_result = pd.concat(results, ignore_index=True)

cv_result.index = cities

cv_result

In [ ]:
print(cv_result.to_latex(caption="Porównanie miasta", float_format="%.3f"))

In [43]:
results = {}

for city, table in geo_city_df_agg.items():
    y1 = table['price'].values
    X = table.drop(columns=['price', 'price_per_m2', 'geometry', 'latitude', 'longitude'], axis=1)

    coords = list(zip(table.geometry.x, table.geometry.y))

    results[city] = Models.evaluate_model(X=X[['squareMeters']], y=y1, coords=coords, models=models)['Test']